# Datathon@IndoML 2026 — Track 2 (Noise Event Removal), end to endTrack 1 is trained here **as the conditioning source for Track 2**, not for its own submission.Sized to finish inside one Kaggle GPU session.- **Track 1** — WavLM-base+ frame encoder (50 Hz = exactly the 20 ms label grid) + BiGRU heads,  trained with a **two-stage schedule** and a mean-teacher semi-supervised loss over the  Gold / Silver / Bronze tiers.- **Bridge** — Track 1 frame posteriors persisted to `t1_cond/*.npz`.- **Track 2** — a mask network built on a **pretrained WavLM encoder**, initialised from the  fine-tuned Track 1 checkpoint so its features are already noise-event aware on Vaani audio.  SSL features *and* Track 1 posteriors are injected by FiLM at two depths, and the whole thing  is optimised directly on SI-SDR.Why reuse the Track 1 encoder rather than load a stock one: it has already been fine-tuned tolocate noise events in exactly this audio, so its 50 Hz frames carry a representation theenhancement head would otherwise have to learn from 4k mixtures. It is also free — the weightsare already in the session. Set `T2_SSL_FROM_T1 = False` to start from stock WavLM instead andmeasure the difference.Track 1 metrics are the official scorer, verbatim, because the quality of the conditioning iswhat Track 2 inherits.## Budget`BUDGET = "fast"` is the default and targets **~3 h end to end** on a T4. `"full"` is ~7 h andneeds a second session. Clips are **5 s** rather than 10 s, which halves the WavLM sequencelength and lets the batch double — the single biggest speed lever available.| stage | fast | full ||---|---|---|| download + extract | ~35 min | ~1 h || Track 1 two-stage training (conditioning source) | ~30 min | ~2 h || sweep + bridge + simulate | ~15 min | ~25 min || **Track 2 training** | ~35 min | ~1 h 10 || Track 2 submission | ~15 min | ~20 min |## Setup1. Accept the terms at `huggingface.co/datasets/ARTPARK-IISc/Vaani-Noise-Event-Dataset`.2. Kaggle → **Add-ons → Secrets** → add `HF_TOKEN`.3. **Internet: ON**, **Accelerator: GPU T4 x2**.4. Optional but recommended: run Sections 1–2 once on a **CPU** session, save the output as a   Kaggle Dataset, then set `RESUME_FROM` and never re-download.

---## 0. Config

In [ ]:
import os, json, math, random, time, io, sys, zipfile
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

BUDGET = "fast"        # "fast" (~3 h) or "full" (~7 h)

CFG = dict(
    sr=16000, n_fft=1024, hop=160, n_mels=64,
    clip_sec=5.0,            # 5 s: halves the WavLM sequence, doubles the batch
    time_pool=2,             # 20 ms frames == WavLM's native 50 Hz
    n_cat=7,
    silver_weight=0.4,
    ema_decay=0.999, max_cons_w=2.0, rampup_epochs=4,
    pos_weight=4.0, mixup_prob=0.5, mixup_alpha=0.2,
    weight_decay=1e-2, val_frac=0.1, seed=42,
)

ENCODER_NAME = "microsoft/wavlm-base-plus"
DROP_LAST_LAYERS = 2       # Schmid et al. ICASSP 2025: final layers overfit the pretrain task
LR_ENC, LR_HEAD = 5e-5, 1e-3

# ---- Track 2 encoder ----
# "sravaani" : ARTPARK-IISc/SraVaani-1.0, 430M FastConformer ASR trained on 31k h of Vaani.
#              In-domain, and ASR features tell the mask net what is phonetically load-bearing
#              - which is the signal that stops over-suppression from deleting phonemes.
#              dWER is half the Track 2 score, so this bias is the right one HERE.
#              Gated: accept the terms at huggingface.co/ARTPARK-IISc/SraVaani-1.0 first.
#              Caveat: FastConformer subsamples 8x -> 80 ms frames, so its features are
#              upsampled to the 20 ms grid. Fine for conditioning, NOT usable for Track 1
#              detection where the tolerance floor is 50 ms.
# "wavlm"    : reuse the fine-tuned Track 1 encoder (20 ms frames, noise-aware). Fallback.
T2_ENCODER     = "sravaani"
SRAVAANI_REPO  = "ARTPARK-IISc/SraVaani-1.0"
T2_SSL_FROM_T1 = True     # only used when T2_ENCODER == "wavlm"
T2_SSL_FROZEN  = True     # frozen either way; 430M is a feature extractor, not trainable here
T2_SSL_PROJ    = 128      # project encoder frames down before conditioning

if BUDGET == "fast":
    MAX_PER_QUALITY = {"verified_timestamps": 6000, "unverified_timestamps": 6000,
                       "no_timestamps": 2000}
    STAGE1_EPOCHS, STAGE2_EPOCHS, T1_BS = 3, 8, 16
    N_MIX, T2_EPOCHS, T2_BS = 4000, 25, 8
else:
    MAX_PER_QUALITY = {"verified_timestamps": None, "unverified_timestamps": 20000,
                       "no_timestamps": 8000}
    STAGE1_EPOCHS, STAGE2_EPOCHS, T1_BS = 6, 16, 16
    N_MIX, T2_EPOCHS, T2_BS = 8000, 40, 8

CATS = ["animal", "vehicle_traffic", "baby_child", "singing_music",
        "phone_signal_alarm", "appliance_machine", "human_non_speech"]
CAT2IDX = {c: i for i, c in enumerate(CATS)}
REPO = "ARTPARK-IISc/Vaani-Noise-Event-Dataset"
WORK = Path("/kaggle/working")
RESUME_FROM = None                      # e.g. Path("/kaggle/input/vaani-prep")

FEAT_DIR = WORK / "feats"; META_PATH = WORK / "meta.jsonl"
CLEAN_DIR = WORK / "bank/clean"; NOISE_DIR = WORK / "bank/noise"
COND_DIR = WORK / "t1_cond"; MIX_DIR = WORK / "t2/mix"; REF_DIR = WORK / "t2/clean"
T2_MANIFEST = WORK / "t2_manifest.jsonl"
CKPT = WORK / "t1_wavlm.pt"; CKPT_T2 = WORK / "t2_sslmask.pt"
if RESUME_FROM is not None:
    FEAT_DIR = RESUME_FROM / "feats"; META_PATH = RESUME_FROM / "meta.jsonl"
    CLEAN_DIR = RESUME_FROM / "bank/clean"; NOISE_DIR = RESUME_FROM / "bank/noise"
for d in [FEAT_DIR, CLEAN_DIR, NOISE_DIR, COND_DIR, MIX_DIR, REF_DIR]:
    try: d.mkdir(parents=True, exist_ok=True)
    except Exception: pass

SIM = dict(dur_sec=CFG["clip_sec"], min_clean_sec=2.5, min_noise_sec=0.3, max_noise_sec=3.0,
           events_per_mix=(1, 2), snr_db=(-5.0, 12.0), seed=1234)
MAX_SPEECH_RATIO = 0.35

N_SAMP = int(CFG["clip_sec"] * CFG["sr"])
FRAMES_100HZ = int(CFG["clip_sec"] * CFG["sr"] / CFG["hop"])
FRAMES_OUT = FRAMES_100HZ // CFG["time_pool"]
FRAME_SEC = CFG["hop"] * CFG["time_pool"] / CFG["sr"]
random.seed(CFG["seed"]); np.random.seed(CFG["seed"])
print(f"BUDGET={BUDGET} | clip {CFG['clip_sec']}s | {FRAMES_OUT} frames @ {FRAME_SEC*1000:.0f} ms")

---## 1. Data — scan shards, then pull only what has gold

In [ ]:
if RESUME_FROM is None:
    !pip -q install datasets huggingface_hub soundfile librosa webrtcvad-wheels transformers 2>/dev/null | tail -1
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login, list_repo_files, hf_hub_download
    from datasets import load_dataset, Audio
    from tqdm.auto import tqdm
    import pyarrow.parquet as pq

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    parquets = sorted([f for f in list_repo_files(REPO, repo_type="dataset")
                       if f.endswith(".parquet")])
    print(len(parquets), "shards")

    QUALS = ["verified_timestamps", "unverified_timestamps", "no_timestamps"]

    def shard_quality_counts(files, token=None):
        try:
            from huggingface_hub import HfFileSystem
            fs = HfFileSystem(token=token)
        except Exception as e:
            print("HfFileSystem unavailable:", e); fs = None
        out = {}
        for f in tqdm(files, desc="scanning"):
            try:
                if os.path.exists(f):
                    col = pq.read_table(f, columns=["annotationQuality"]).column(0).to_pylist()
                elif fs is not None:
                    with fs.open(f"datasets/{REPO}/{f}") as fh:
                        col = pq.read_table(fh, columns=["annotationQuality"]).column(0).to_pylist()
                else:
                    raise RuntimeError("no fs")
                out[f] = Counter(col)
            except Exception as e:
                out[f] = None
        return out

    def pick_shards(counts, targets):
        ok = {f: c for f, c in counts.items() if c}
        chosen, got = [], Counter()
        for q in QUALS:
            tgt = targets.get(q) or 0
            if tgt == 0 and targets.get(q) is not None:
                continue
            tgt = tgt or 10 ** 9
            for f, c in sorted(ok.items(), key=lambda kv: -kv[1].get(q, 0)):
                if got[q] >= tgt: break
                if f in chosen or c.get(q, 0) == 0: continue
                chosen.append(f); got.update(c)
        return chosen, got

    counts = shard_quality_counts(parquets, token=HF_TOKEN)
    scanned = {f: c for f, c in counts.items() if c}
    if scanned:
        tot = Counter()
        for c in scanned.values(): tot.update(c)
        print("\ntotals:", {q: tot.get(q, 0) for q in QUALS})
        SHARDS, got = pick_shards(counts, MAX_PER_QUALITY)
        print(f"selected {len(SHARDS)} shards ->", {q: got.get(q, 0) for q in QUALS})
        if got.get(QUALS[0], 0) < 1000:
            print("!! under 1000 gold selected - Track 1 will be weak and the Track 2 banks thin")
    else:
        SHARDS = parquets[:8]
        print("\nscan failed everywhere; falling back to first 8 shards")
else:
    print("RESUME_FROM set - skipping Sections 1-2")

In [ ]:
if RESUME_FROM is None:
    LOCAL = Path("/kaggle/temp/parquet"); LOCAL.mkdir(parents=True, exist_ok=True)
    local_files = [hf_hub_download(REPO, f, repo_type="dataset", local_dir=str(LOCAL),
                                   token=HF_TOKEN) for f in tqdm(SHARDS, desc="download")]
    raw = load_dataset("parquet", data_files=local_files, split="train")
    # Never let `datasets` decode audio: the Audio feature changed twice (mono -> num_channels,
    # and decoding now needs torchcodec and returns an AudioDecoder). decode=False works on
    # every version and hands back bytes we decode ourselves.
    try:
        raw = raw.cast_column("audio", Audio(decode=False)); print("decode=False ok")
    except Exception as e:
        print("decode=False failed, will fall back per row:", e)
    print(raw)
    print("mix:", Counter(raw["annotationQuality"]))
    ex0 = raw[0]
    print("timestamps sample:", ex0.get("NoiseSubCategoryTimeStamp"))
    print("NOTE: empty timestamps on a `no_timestamps` row is CORRECT - bronze has tags only.")

---## 2. One pass over the audioProduces everything both tracks need, decoding each clip exactly once:| output | tiers | used by ||---|---|---|| `feats/*.npz` — int16 waveform + `(8, T)` labels | all | Track 1 || `bank/clean/*.wav` — noise-free spans | gold | Track 2 clean reference || `bank/noise/*.wav` — VAD-filtered event spans | gold | Track 2 noise source |The clean bank is the *complement* of annotated events, so it is **pseudo**-clean, not clean.The noise bank is VAD-filtered because unfiltered "noise" is half speech, which would make theSI-SDR target wrong and teach the enhancer to delete words.

In [ ]:
import torch, torchaudio, soundfile as sf, librosa
import torch.nn as nn, torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

# Seed torch too. random/np were seeded in Section 0, but model init and the DataLoader
# sampler draw from torch's generator, which otherwise starts from OS entropy - so two runs
# of the same config would not be comparable. DataLoader workers inherit from this seed.
torch.manual_seed(CFG["seed"]); torch.cuda.manual_seed_all(CFG["seed"])
print("seeded torch:", CFG["seed"])

def decode_audio(a, target_sr=CFG["sr"]):
    w = sr = None
    if isinstance(a, dict):
        if a.get("array") is not None:
            w, sr = np.asarray(a["array"], dtype=np.float32), a["sampling_rate"]
        elif a.get("bytes"):
            w, sr = sf.read(io.BytesIO(a["bytes"]), dtype="float32", always_2d=False)
        elif a.get("path"):
            w, sr = sf.read(a["path"], dtype="float32", always_2d=False)
    elif hasattr(a, "get_all_samples"):
        s = a.get_all_samples(); w, sr = s.data.numpy().astype(np.float32), int(s.sample_rate)
    elif hasattr(a, "path"):
        w, sr = sf.read(a.path, dtype="float32", always_2d=False)
    if w is None:
        raise ValueError(f"cannot decode {type(a)}")
    w = np.asarray(w, dtype=np.float32)
    if w.ndim > 1:
        w = w.mean(axis=0) if w.shape[0] < w.shape[1] else w.mean(axis=1)
    if sr != target_sr:
        w = librosa.resample(w, orig_sr=sr, target_sr=target_sr)
    return np.ascontiguousarray(w, dtype=np.float32)

def _f(x):
    try: return float(x)
    except: return None

def spans_from(ex):
    out = []
    for s in (ex.get("NoiseSubCategoryTimeStamp") or []):
        st, en = _f(s.get("start")), _f(s.get("end"))
        if st is not None and en is not None and en > st:
            out.append((st, en, s.get("category")))
    return sorted(out)

def build_labels(spans, n_frames, has_strong):
    lab = np.zeros((1 + CFG["n_cat"], n_frames), dtype=np.uint8)
    if has_strong:
        for st, en, cat in spans:
            a, b = max(0, int(round(st * 100))), min(n_frames, int(round(en * 100)))
            if b <= a: continue
            lab[0, a:b] = 1
            ci = CAT2IDX.get(cat)
            if ci is not None: lab[1 + ci, a:b] = 1
    return lab

def clip_tags(ex):
    y = np.zeros(1 + CFG["n_cat"], dtype=np.float32)
    cats = ex.get("NoiseCategory") or []
    if len(cats): y[0] = 1.0
    for c in cats:
        ci = CAT2IDX.get(c)
        if ci is not None: y[1 + ci] = 1.0
    return y

def complement(spans, dur, pad=0.05):
    free, cur = [], 0.0
    for st, en, _ in spans:
        st, en = max(0.0, st - pad), min(dur, en + pad)
        if st > cur: free.append((cur, st))
        cur = max(cur, en)
    if cur < dur: free.append((cur, dur))
    return free

try:
    import webrtcvad; _vad = webrtcvad.Vad(2); HAVE_VAD = True
except Exception:
    HAVE_VAD = False

def speech_ratio(wav, sr=CFG["sr"]):
    if HAVE_VAD:
        pcm = np.clip(wav * 32767, -32768, 32767).astype(np.int16).tobytes()
        n = int(sr * 0.03) * 2
        fr = [pcm[i:i+n] for i in range(0, len(pcm) - n + 1, n)]
        return float(np.mean([_vad.is_speech(f, sr) for f in fr])) if fr else 1.0
    S = np.abs(np.fft.rfft(wav * np.hanning(len(wav)))) ** 2
    f = np.fft.rfftfreq(len(wav), 1 / sr)
    return float(S[(f >= 300) & (f <= 3400)].sum() / (S.sum() + 1e-9))

In [ ]:
if RESUME_FROM is None:
    counts_done = {k: 0 for k in MAX_PER_QUALITY}
    clean_idx, noise_idx = [], []
    mf = open(META_PATH, "w"); t0 = time.time()

    def done():
        for q, cap in MAX_PER_QUALITY.items():
            if cap is None: return False
            if cap and counts_done[q] < cap: return False
        return True

    for i, ex in enumerate(tqdm(raw)):
        q = ex["annotationQuality"]; cap = MAX_PER_QUALITY.get(q, 0)
        if cap == 0: continue
        if cap is not None and counts_done[q] >= cap:
            if done(): break
            continue
        try:
            wav = decode_audio(ex["audio"])
        except Exception as e:
            if i < 5: print("decode failed row", i, e)
            continue
        if wav is None or len(wav) < CFG["sr"] * 0.2: continue
        dur = len(wav) / CFG["sr"]
        spans = spans_from(ex)
        has_strong = q in ("verified_timestamps", "unverified_timestamps")

        nf = 1 + len(wav) // CFG["hop"]
        key = f"{q[:3]}_{i:07d}"
        np.savez_compressed(FEAT_DIR / f"{key}.npz",
                            wav=(np.clip(wav, -1, 1) * 32767).astype(np.int16),
                            lab=build_labels(spans, nf, has_strong), tags=clip_tags(ex))
        mf.write(json.dumps({"key": key, "quality": q, "n_frames": int(nf),
                             "has_strong": bool(has_strong),
                             "language": ex.get("language")}) + "\n")
        counts_done[q] += 1

        if q != "verified_timestamps": continue
        for k, (a, b) in enumerate(complement(spans, dur)):
            if b - a < SIM["min_clean_sec"]: continue
            seg = wav[int(a * CFG["sr"]):int(b * CFG["sr"])]
            if np.abs(seg).max() < 1e-3: continue
            p = CLEAN_DIR / f"c{i:07d}_{k}.wav"; sf.write(p, seg, CFG["sr"])
            clean_idx.append({"path": str(p), "dur": len(seg) / CFG["sr"]})
        for k, (a, b, cat) in enumerate(spans):
            d = min(b, dur) - max(0.0, a)
            if not (SIM["min_noise_sec"] <= d <= SIM["max_noise_sec"]): continue
            seg = wav[int(a * CFG["sr"]):int(b * CFG["sr"])]
            if np.abs(seg).max() < 1e-3 or speech_ratio(seg) > MAX_SPEECH_RATIO: continue
            p = NOISE_DIR / f"n{i:07d}_{k}.wav"; sf.write(p, seg, CFG["sr"])
            noise_idx.append({"path": str(p), "dur": len(seg) / CFG["sr"],
                              "cat": CAT2IDX.get(cat, -1)})

    mf.close()
    json.dump(clean_idx, open(WORK / "clean_index.json", "w"))
    json.dump(noise_idx, open(WORK / "noise_index.json", "w"))
    print(counts_done, f"| {(time.time()-t0)/60:.1f} min")
    print(f"clean {len(clean_idx)} ({sum(c['dur'] for c in clean_idx)/3600:.2f} h) | "
          f"noise {len(noise_idx)} ({sum(n['dur'] for n in noise_idx)/3600:.2f} h)")
    if counts_done.get("verified_timestamps", 0) == 0:
        print("!! ZERO gold - go back to Section 1 and select shards with verified_timestamps")
    if clean_idx:
        d = np.array([c["dur"] for c in clean_idx])
        print("clean span dur p10/p50/p90: %.2f %.2f %.2f s" % tuple(np.percentile(d, [10, 50, 90])))
    print(Counter([CATS[n["cat"]] if n["cat"] >= 0 else "?" for n in noise_idx]))
else:
    from tqdm.auto import tqdm
    clean_idx = json.load(open(RESUME_FROM / "clean_index.json"))
    noise_idx = json.load(open(RESUME_FROM / "noise_index.json"))
    print(f"resumed {len(clean_idx)} clean, {len(noise_idx)} noise")

---## 3. Track 1 dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

meta = [json.loads(l) for l in open(META_PATH)]
meta = [m for m in meta if (FEAT_DIR / f"{m['key']}.npz").exists()]
gold = [m for m in meta if m["quality"] == "verified_timestamps"]
silver = [m for m in meta if m["quality"] == "unverified_timestamps"]
bronze = [m for m in meta if m["quality"] == "no_timestamps"]
print(f"gold {len(gold)} | silver {len(silver)} | bronze {len(bronze)}")

random.Random(CFG["seed"]).shuffle(gold)
n_val = max(1, int(len(gold) * CFG["val_frac"]))
val_meta, train_meta = gold[:n_val], gold[n_val:] + silver + bronze
random.Random(CFG["seed"]).shuffle(train_meta)
print(f"train {len(train_meta)} | val {len(val_meta)} (gold only)")

def prep_wav(w):
    return (w - w.mean()) / (w.std() + 1e-5)

class SEDDataset(Dataset):
    def __init__(self, items, train=True):
        self.items, self.train = items, train
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        m = self.items[i]
        d = np.load(FEAT_DIR / f"{m['key']}.npz")
        wav = d["wav"].astype(np.float32) / 32767.0
        lab = d["lab"].astype(np.float32); tags = d["tags"].astype(np.float32)
        if len(wav) > N_SAMP:
            s = random.randint(0, len(wav) - N_SAMP) if self.train else 0
            wav = wav[s:s + N_SAMP]
            lab = lab[:, s // CFG["hop"]: s // CFG["hop"] + FRAMES_100HZ]
            valid = N_SAMP
        else:
            valid = len(wav); wav = np.pad(wav, (0, N_SAMP - len(wav)))
        if lab.shape[1] < FRAMES_100HZ:
            lab = np.pad(lab, ((0, 0), (0, FRAMES_100HZ - lab.shape[1])))
        lab = lab[:, :FRAMES_100HZ]
        lab_o = lab.reshape(lab.shape[0], FRAMES_OUT, CFG["time_pool"]).max(axis=2)
        mask = np.zeros(FRAMES_OUT, dtype=np.float32)
        mask[:max(1, int(math.ceil(valid / CFG["sr"] / FRAME_SEC)))] = 1.0
        # A crop can cut the event out of the window while the stored clip tag still claims
        # it is present; that contradiction poisons the weak loss. Bronze has no timestamps.
        if m["has_strong"]:
            tags = lab_o.max(axis=1)
        return dict(wav=torch.from_numpy(prep_wav(wav)), lab=torch.from_numpy(lab_o),
                    tags=torch.from_numpy(tags), mask=torch.from_numpy(mask),
                    strong=torch.tensor(1.0 if m["has_strong"] else 0.0),
                    w=torch.tensor(CFG["silver_weight"]
                                   if m["quality"] == "unverified_timestamps" else 1.0))

train_dl = DataLoader(SEDDataset(train_meta), batch_size=T1_BS, shuffle=True, num_workers=2,
                      pin_memory=True, drop_last=True, persistent_workers=True)
val_dl = DataLoader(SEDDataset(val_meta, train=False), batch_size=T1_BS, shuffle=False,
                    num_workers=2)
b = next(iter(train_dl))
print({k: tuple(v.shape) for k, v in b.items()})
print(f"steps/epoch {len(train_dl)}")

---## 4. Track 1 model — WavLM-base+ frame encoderWavLM outputs 50 Hz, exactly the 20 ms label grid, so no interpolation is needed in training.It was also pretrained with simulated noise and overlapped speech, so its representationsencode background events rather than discarding them.

In [ ]:
from transformers import AutoModel

class WavLMSED(nn.Module):
    input_key = "wav"
    def __init__(self, name=ENCODER_NAME, n_out=8, rnn_dim=256, drop_layers=DROP_LAST_LAYERS):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        if drop_layers > 0:
            self.enc.encoder.layers = self.enc.encoder.layers[:-drop_layers]
        d = self.enc.config.hidden_size
        self.rnn = nn.GRU(d, rnn_dim, 2, batch_first=True, bidirectional=True, dropout=0.1)
        self.strong = nn.Linear(2 * rnn_dim, n_out)
        self.att = nn.Linear(2 * rnn_dim, n_out)
    def forward(self, x, mask=None):
        h = self.enc(x).last_hidden_state
        T = mask.shape[1] if mask is not None else h.shape[1]
        if h.shape[1] != T:                       # conv stride rounding, off by a frame or two
            h = F.interpolate(h.transpose(1, 2), size=T, mode="linear",
                              align_corners=False).transpose(1, 2)
        h, _ = self.rnn(h)
        frame = torch.sigmoid(self.strong(h))
        a = self.att(h)
        if mask is not None:
            a = a.masked_fill(mask.unsqueeze(-1) < 0.5, -1e4)
        a = torch.softmax(a, dim=1)
        clip = (frame * a).sum(dim=1).clamp(1e-6, 1 - 1e-6)
        return frame.transpose(1, 2), clip

_m = WavLMSED().to(device)
print(f"{sum(p.numel() for p in _m.parameters())/1e6:.1f}M params")
with torch.no_grad():
    _f_, _c_ = _m(b["wav"].to(device), b["mask"].to(device))
print("frame", tuple(_f_.shape), "clip", tuple(_c_.shape))
del _m; torch.cuda.empty_cache()

---## 5. Metrics — official scorer, verbatimCopied from the Evaluation page. Do not "improve" these; the point is that local == leaderboard.Three things that drive tuning:- tolerance is `max(0.20 × d, 0.05)` — the floor is **50 ms**- matching is **global closest-first**, not first-fit- Dice is macro per clip, and **a clip with no reference and no prediction scores 1.0** — so a  single false alarm on a clean clip costs that clip's entire Dice point**Combined = Event F1 + Segment Dice**, max 2.0.

In [ ]:
from scipy.ndimage import median_filter

def match_events(ref_events, pred_events, tolerance_frac=0.20):
    matched_ref, matched_pred, candidates = set(), set(), []
    for ri, (r_on, r_off) in enumerate(ref_events):
        tol = max(tolerance_frac * (r_off - r_on), 0.05)
        for pi, (p_on, p_off) in enumerate(pred_events):
            if abs(p_on - r_on) <= tol and abs(p_off - r_off) <= tol:
                candidates.append((abs(p_on - r_on) + abs(p_off - r_off), ri, pi))
    for _, ri, pi in sorted(candidates):
        if ri not in matched_ref and pi not in matched_pred:
            matched_ref.add(ri); matched_pred.add(pi)
    tp = len(matched_ref)
    return tp, len(pred_events) - tp, len(ref_events) - tp

def event_based_f1(ref_data, pred_data):
    TP = FP = FN = 0
    for cid, ref_events in ref_data.items():
        tp, fp, fn = match_events(ref_events, pred_data.get(cid, []))
        TP += tp; FP += fp; FN += fn
    for cid in pred_data:
        if cid not in ref_data: FP += len(pred_data[cid])
    p = TP / (TP + FP) if (TP + FP) else 0.0
    r = TP / (TP + FN) if (TP + FN) else 0.0
    return ((2 * p * r / (p + r)) if (p + r) else 0.0), p, r, TP, FP, FN

def events_to_frames(events, max_time, frame_len=0.01):
    n = int(max_time / frame_len) + 1
    mask = [0] * n
    for on, off in events:
        for i in range(int(on / frame_len), min(int(off / frame_len) + 1, n)):
            mask[i] = 1
    return mask

def segment_dice(ref_data, pred_data):
    scores = []
    for cid, ref_events in ref_data.items():
        pred_events = pred_data.get(cid, [])
        all_ev = ref_events + pred_events
        if not all_ev:
            scores.append(1.0); continue
        max_time = max(off for _, off in all_ev) + 0.5
        rm, pm = events_to_frames(ref_events, max_time), events_to_frames(pred_events, max_time)
        inter = sum(r & p for r, p in zip(rm, pm)); total = sum(rm) + sum(pm)
        scores.append(1.0 if total == 0 else 2.0 * inter / total)
    return sum(scores) / len(scores) if scores else 0.0

def prob_to_events(p, thr=0.5, med=7, frame_sec=FRAME_SEC, min_dur=0.05, merge_gap=0.10):
    b_ = (np.asarray(p) >= thr).astype(np.uint8)
    if med > 1: b_ = median_filter(b_, size=med, mode="nearest")
    ev, start = [], None
    for i, v in enumerate(b_):
        if v and start is None: start = i
        elif not v and start is not None:
            ev.append([start * frame_sec, i * frame_sec]); start = None
    if start is not None: ev.append([start * frame_sec, len(b_) * frame_sec])
    out = []
    for e in ev:
        if out and e[0] - out[-1][1] <= merge_gap: out[-1][1] = e[1]
        else: out.append(e)
    return [e for e in out if e[1] - e[0] >= min_dur]

@torch.no_grad()
def cache_posteriors(model, dl):
    # posteriors do not change when sweeping post-processing, so run the model ONCE
    model.eval(); cache = []
    for batch in dl:
        frame, clip = model(batch["wav"].to(device), batch["mask"].to(device))
        for p, cp, g, msk in zip(frame[:, 0].float().cpu().numpy(),
                                 clip[:, 0].float().cpu().numpy(),
                                 batch["lab"][:, 0].numpy(), batch["mask"].numpy()):
            nv = int(msk.sum()); cache.append((p[:nv].copy(), g[:nv].copy(), float(cp)))
    ref = {f"c{i}": [tuple(e) for e in prob_to_events(g, 0.5, 1, min_dur=0.0, merge_gap=0.0)]
           for i, (_, g, _) in enumerate(cache)}
    return cache, ref

def score_cached(cache, ref, thr=0.5, med=7, min_dur=0.05, merge_gap=0.10, clip_gate=0.0):
    # clip_gate: if the attention-pooled clip probability is below this, emit NO events.
    # Dice is macro per clip and a clip with no reference and no prediction scores 1.0, so
    # staying silent on a clip you believe is clean buys a whole Dice point. Suppressing a
    # real event costs only its share of F1. The asymmetry is large; sweep the gate.
    pred = {}
    for i, (p, _, cp) in enumerate(cache):
        pred[f"c{i}"] = ([] if cp < clip_gate else
                         [tuple(e) for e in prob_to_events(p, thr, med, min_dur=min_dur,
                                                           merge_gap=merge_gap)])
    f1, prec, rec, TP, FP, FN = event_based_f1(ref, pred)
    dice = segment_dice(ref, pred)
    return dict(f1=f1, dice=dice, score=f1 + dice, precision=prec, recall=rec,
                tp=TP, fp=FP, fn=FN)

def evaluate(model, dl, **kw):
    cache, ref = cache_posteriors(model, dl)
    return score_cached(cache, ref, **kw)

---## 6. Two-stage fine-tuning**Stage 1** freezes the encoder and trains only the GRU + heads. **Stage 2** unfreezes with amuch smaller LR on the encoder than the heads. This is not optional: fine-tuning a largepretrained encoder end-to-end from step 0 overfits immediately and lands *below* a from-scratchCRNN. That failure is the reason the ATST-SED paper exists.Also active: mean teacher with a ramped EMA decay, positive-weighted BCE (noise frames are asmall minority), and mixup with **union** labels.Losses are computed in fp32 outside the autocast region — `BCELoss` is explicitly unsafe toautocast, and an fp16 sigmoid can saturate to exactly 1.0.

In [ ]:
import copy
from torch.amp import autocast, GradScaler

def wav_augment(x, n=2, tmax=None):
    tmax = tmax or int(0.15 * CFG["sr"])
    x = x.clone()
    for b_ in range(x.shape[0]):
        for _ in range(n):
            t = random.randint(0, tmax); t0 = random.randint(0, max(0, x.shape[1] - t))
            x[b_, t0:t0 + t] = 0
    return x

def mixup(inp, lab, tags, alpha=CFG["mixup_alpha"]):
    lam = float(np.random.beta(alpha, alpha)); lam = max(lam, 1 - lam)
    perm = torch.randperm(inp.size(0), device=inp.device)
    return (lam * inp + (1 - lam) * inp[perm],
            torch.maximum(lab, lab[perm]), torch.maximum(tags, tags[perm]))

def bce_pos(p, target, pw=None):
    pw = CFG["pos_weight"] if pw is None else pw
    p = p.clamp(1e-6, 1 - 1e-6)
    return -(pw * target * torch.log(p) + (1 - target) * torch.log(1 - p))

def rampup(ep, n):
    return 1.0 if n == 0 else float(np.exp(-5 * (1 - np.clip(ep / n, 0, 1)) ** 2))

@torch.no_grad()
def ema_update(stu, tea, decay, step=None):
    if step is not None:
        decay = min(1 - 1 / (step + 1), decay)      # else the teacher stays at random init
    for ts, ss in zip(tea.state_dict().values(), stu.state_dict().values()):
        if ts.dtype.is_floating_point: ts.mul_(decay).add_(ss.detach(), alpha=1 - decay)
        else: ts.copy_(ss)

state = dict(best=-1.0, gstep=0)

def run_epochs(stu, tea, opt, sched, epochs, scaler, tag=""):
    for ep in range(epochs):
        stu.train(); tea.train()
        cw = CFG["max_cons_w"] * rampup(ep, CFG["rampup_epochs"])
        acc = defaultdict(float); t0 = time.time()
        for batch in train_dl:
            inp = batch["wav"].to(device, non_blocking=True)
            lab = batch["lab"].to(device); tags = batch["tags"].to(device)
            mask = batch["mask"].to(device)
            strong = batch["strong"].to(device); w = batch["w"].to(device)
            if random.random() < CFG["mixup_prob"]:
                inp, lab, tags = mixup(inp, lab, tags)
            with autocast("cuda"):
                f_s, c_s = stu(wav_augment(inp), mask)
                with torch.no_grad():
                    f_t, c_t = tea(inp, mask)
            m3 = mask.unsqueeze(1)
            f_s = f_s.float().clamp(1e-6, 1 - 1e-6); c_s = c_s.float().clamp(1e-6, 1 - 1e-6)
            ls = (bce_pos(f_s, lab) * m3).sum(dim=(1, 2)) / (m3.sum(dim=(1, 2)) * lab.shape[1] + 1e-6)
            l_strong = (ls * strong * w).sum() / (strong * w).sum().clamp(min=1e-6)
            l_weak = bce_pos(c_s, tags).mean()
            f_t, c_t = f_t.float(), c_t.float()
            l_cons = (((f_s - f_t) ** 2) * m3).sum() / (m3.sum() * lab.shape[1] + 1e-6) \
                     + ((c_s - c_t) ** 2).mean()
            loss = l_strong + l_weak + cw * l_cons
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(stu.parameters(), 5.0)
            scaler.step(opt); scaler.update()
            if sched is not None: sched.step()
            state["gstep"] += 1
            ema_update(stu, tea, CFG["ema_decay"], step=state["gstep"])
            acc["s"] += float(l_strong.detach()); acc["w"] += float(l_weak.detach())
            acc["c"] += float(l_cons.detach()); acc["n"] += 1
        n = max(1, acc["n"])
        ms, mt = evaluate(stu, val_dl), evaluate(tea, val_dl)
        print(f"{tag}ep{ep:02d} [{(time.time()-t0)/60:.1f}m] s {acc['s']/n:.3f} w {acc['w']/n:.3f} "
              f"c {acc['c']/n:.3f} | stu {ms['score']:.4f} (F1 {ms['f1']:.3f} D {ms['dice']:.3f}) "
              f"| tea {mt['score']:.4f}")
        if max(ms["score"], mt["score"]) > state["best"]:
            state["best"] = max(ms["score"], mt["score"])
            which = "student" if ms["score"] >= mt["score"] else "teacher"
            torch.save({"model": (stu if which == "student" else tea).state_dict(),
                        "which": which, "score": state["best"], "cfg": CFG}, CKPT)
            print(f"   saved {which} {state['best']:.4f}")

student = WavLMSED().to(device)
teacher = copy.deepcopy(student).to(device)
for p in teacher.parameters(): p.requires_grad_(False)
scaler = GradScaler("cuda")
head = [p for n_, p in student.named_parameters() if not n_.startswith("enc.")]

print(f"=== stage 1: encoder frozen, {STAGE1_EPOCHS} epochs ===")
for p in student.enc.parameters(): p.requires_grad_(False)
run_epochs(student, teacher,
           torch.optim.AdamW(head, lr=LR_HEAD, weight_decay=CFG["weight_decay"]),
           None, STAGE1_EPOCHS, scaler, tag="s1 ")

print(f"=== stage 2: unfrozen, enc lr {LR_ENC}, {STAGE2_EPOCHS} epochs ===")
for p in student.enc.parameters(): p.requires_grad_(True)
opt2 = torch.optim.AdamW([{"params": student.enc.parameters(), "lr": LR_ENC},
                          {"params": head, "lr": LR_HEAD * 0.3}],
                         weight_decay=CFG["weight_decay"])
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=STAGE2_EPOCHS * max(1, len(train_dl)))
run_epochs(student, teacher, opt2, sched2, STAGE2_EPOCHS, scaler, tag="s2 ")
print("\nbest Combined:", state["best"], "/ 2.0")
del student, teacher; torch.cuda.empty_cache()

---## 7. Post-processing sweepOne forward pass over val, then every config scored on the cached posteriors. Precision mattersmore than it looks here: Dice is macro per clip, so a false alarm on a clean clip costs thatclip's whole point.

In [ ]:
ck = torch.load(CKPT, map_location=device, weights_only=False)
t1 = WavLMSED().to(device); t1.load_state_dict(ck["model"]); t1.eval()
print("loaded", ck["which"], f"{ck['score']:.4f}")

t0 = time.time(); cache, ref = cache_posteriors(t1, val_dl)
print(f"cached {len(cache)} clips in {time.time()-t0:.0f}s")

n_empty_ref = sum(1 for v in ref.values() if not v)
print(f"val clips with NO reference events: {n_empty_ref}/{len(ref)} "
      f"({100*n_empty_ref/max(1,len(ref)):.1f}%) - each is a free Dice point if we stay silent")

results = []
for thr in [0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7]:
    for med in [1, 3, 5, 9, 15, 21]:
        for min_dur in [0.05, 0.12, 0.25]:
            for gate in [0.0, 0.3, 0.5, 0.7]:
                m = score_cached(cache, ref, thr=thr, med=med, min_dur=min_dur, clip_gate=gate)
                results.append((m["score"], m["f1"], m["dice"], thr, med, min_dur, gate,
                                m["precision"], m["recall"]))
results.sort(reverse=True)
print(f"{'Comb':>7}{'F1':>7}{'Dice':>7}{'thr':>6}{'med':>5}{'mind':>7}{'gate':>6}{'P':>7}{'R':>7}")
for r in results[:12]:
    print(f"{r[0]:7.4f}{r[1]:7.3f}{r[2]:7.3f}{r[3]:6.2f}{r[4]:5d}{r[5]:7.2f}{r[6]:6.2f}"
          f"{r[7]:7.3f}{r[8]:7.3f}")

T1_THR, T1_MED, T1_MINDUR, T1_GATE = (results[0][3], results[0][4], results[0][5], results[0][6])
ck.update(thr=float(T1_THR), med=int(T1_MED), min_dur=float(T1_MINDUR), gate=float(T1_GATE))
torch.save(ck, CKPT)
print(f"\nthr={T1_THR} med={T1_MED} min_dur={T1_MINDUR} gate={T1_GATE} "
      f"-> Combined {results[0][0]:.4f} / 2.0")
no_gate = max(r[0] for r in results if r[6] == 0.0)
print(f"gate contributes +{results[0][0]-no_gate:.4f} over the best ungated config")
if results[0][6] < results[0][7] - 0.1:
    print("!! precision << recall: you are firing on quiet clips, which costs a full Dice "
          "point per clean clip. Prefer a higher thr even at some F1 cost.")

---## 8. The bridge — persist Track 1 output for Track 2```t1_cond/{clip_id}.npz    post  float16 (8, T)   posteriors @ 20 ms; ch0 = any-noise, ch1..7 = category    events float32 (N,2)   onset/offset seconds    event_cats int8 (N,)   category index, -1 if unknown```Posteriors, not just thresholded spans — the Track 2 model conditions on the soft signal.

In [ ]:
MAX_ONESHOT_SEC = 120.0

MIN_SAMPLES = 640          # WavLM conv needs >= 400 samples; keep a margin

@torch.no_grad()
def t1_posteriors(wav, sr=CFG["sr"], model=None):
    # whole-clip forward; normalisation matches training exactly.
    # returns (post (8, T20), clip_prob_any_noise)
    model = t1 if model is None else model
    wav = np.asarray(wav, dtype=np.float32)
    if len(wav) < MIN_SAMPLES:
        wav = np.pad(wav, (0, MIN_SAMPLES - len(wav)))
    dur = len(wav) / sr
    n_out = max(1, int(math.ceil(dur / FRAME_SEC)))
    if dur <= MAX_ONESHOT_SEC:
        x = torch.from_numpy(prep_wav(wav)).unsqueeze(0).to(device)
        fr, cl = model(x, None)
        p = fr[0].float().cpu().numpy(); cp = float(cl[0, 0])
        if p.shape[1] != n_out:
            p = np.stack([np.interp(np.linspace(0, 1, n_out),
                                    np.linspace(0, 1, p.shape[1]), r) for r in p])
        return p, cp
    win, hp = int(30 * sr), int(28 * sr)
    starts = list(range(0, max(1, len(wav) - win + hp), hp)) or [0]
    acc = np.zeros((8, n_out)); cnt = np.zeros(n_out) + 1e-9; cps = []
    for s0 in starts:
        seg = wav[s0:s0 + win]
        if len(seg) < MIN_SAMPLES: continue
        x = torch.from_numpy(prep_wav(seg)).unsqueeze(0).to(device)
        fr, cl = model(x, None)
        p = fr[0].float().cpu().numpy(); cps.append(float(cl[0, 0]))
        off = int(round(s0 / sr / FRAME_SEC))
        for j in range(p.shape[1]):
            k = off + j
            if 0 <= k < n_out: acc[:, k] += p[:, j]; cnt[k] += 1
    return acc / cnt, (max(cps) if cps else 0.0)

def export_track1(clip_id, wav, sr=CFG["sr"], cond_dir=COND_DIR, thr=None, med=None,
                  min_dur=None, gate=None):
    thr = T1_THR if thr is None else thr
    med = T1_MED if med is None else med
    min_dur = T1_MINDUR if min_dur is None else min_dur
    gate = T1_GATE if gate is None else gate
    post, clip_p = t1_posteriors(wav, sr)
    dur = max(len(wav) / sr, MIN_SAMPLES / sr)
    spans, cats, scores = [], [], []
    # Clip-level gate: believe the clip is clean -> emit nothing -> full Dice point.
    events_iter = [] if clip_p < gate else prob_to_events(post[0], thr, med, min_dur=min_dur)
    for on, off in events_iter:
        on, off = float(max(0.0, on)), float(min(dur, off))
        if off - on < 0.05: continue
        a, b_ = int(on / FRAME_SEC), max(int(on / FRAME_SEC) + 1, int(off / FRAME_SEC))
        seg = post[1:, a:b_]
        spans.append([on, off])
        cats.append(int(seg.mean(axis=1).argmax()) if seg.size else -1)
        scores.append(float(post[0, a:b_].mean()) if b_ > a else 0.0)
    spans = np.asarray(spans, dtype=np.float32).reshape(-1, 2)
    np.savez_compressed(Path(cond_dir) / f"{clip_id}.npz", post=post.astype(np.float16),
                        events=spans, event_cats=np.asarray(cats, dtype=np.int8),
                        event_score=np.asarray(scores, dtype=np.float32),
                        frame_sec=np.float32(FRAME_SEC), dur=np.float32(dur))
    return {"clip_id": clip_id,
            "events": [{"onset": round(float(s[0]), 3), "offset": round(float(s[1]), 3)}
                       for s in spans],
            "cats": cats}

def load_cond(clip_id, cond_dir=COND_DIR):
    d = np.load(Path(cond_dir) / f"{clip_id}.npz")
    return dict(post=d["post"].astype(np.float32), events=d["events"],
                cats=d["event_cats"], dur=float(d["dur"]))

for nm, _w in [("3s", np.random.randn(CFG["sr"] * 3).astype(np.float32) * 0.05),
               ("0.3s", np.random.randn(int(CFG["sr"] * 0.3)).astype(np.float32) * 0.05),
               ("silence", np.zeros(CFG["sr"] * 2, dtype=np.float32))]:
    r = export_track1(f"__smoke_{nm}__", _w)
    print(f"{nm:>8}: {len(r['events'])} events")
    os.remove(COND_DIR / f"__smoke_{nm}__.npz")

---## 9. Track 1 submissionOfficial format: ZIP with `predictions.jsonl` **at the root**, key `clip_id`, events carrying onlyonset/offset (Track 1 is class-agnostic). Every eval clip appears exactly once; clips with nothingdetected get `[]`, not a missing line. Extra clip_ids not in the reference count all their eventsas false positives. Limits: 5 submissions/day, 100 total.We still predict a category internally (Track 2 uses it); it just does not go in the file.

In [ ]:
AUD = (".wav", ".flac", ".mp3", ".ogg")

def read_audio(p):
    w, sr = sf.read(str(p), dtype="float32", always_2d=False)
    if w.ndim > 1: w = w.mean(axis=1)
    if sr != CFG["sr"]: w = librosa.resample(w, orig_sr=sr, target_sr=CFG["sr"])
    return w

T1_TEST_DIR = Path("/kaggle/input/indoml-track1-test")   # input_data from the Files tab
T1_JSONL, T1_ZIP = WORK / "predictions.jsonl", WORK / "submission_track1.zip"

if T1_TEST_DIR.exists():
    files = sorted([p for p in T1_TEST_DIR.rglob("*") if p.suffix.lower() in AUD])
    print(len(files), "test clips")
    with open(T1_JSONL, "w", encoding="utf-8") as f:
        for p in tqdm(files):
            rec = export_track1(p.stem, read_audio(p))
            f.write(json.dumps({"clip_id": p.stem, "events": rec["events"]},
                               ensure_ascii=False) + "\n")
    with zipfile.ZipFile(T1_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(T1_JSONL, "predictions.jsonl")          # at ZIP ROOT, no enclosing folder
    n = sum(1 for _ in open(T1_JSONL))
    empty = sum(1 for l in open(T1_JSONL) if not json.loads(l)["events"])
    print(f"{n} lines ({empty} with no events) -> {T1_ZIP}")
    print(open(T1_JSONL).readline().strip()[:140])
    print("!! verify clip_id matches the eval metadata; we use the filename stem")
else:
    print(f"{T1_TEST_DIR} not found. Accept terms under 'My Submissions' on Codabench, "
          f"download input_data from the Files tab, attach as a Kaggle Dataset, re-run.")
    print("(read_audio is defined above, so the Track 2 cells below still work.)")

---## 10. Track 2 - simulate paired mixturesSI-SDR needs a clean reference, and 4 of the 11 test hours are themselves synthetic (cleanspeech with noise added), so building data the same way is directly on-distribution.One clean span from **one** source clip — splicing different speakers would make dWER meaninglessand teach the enhancer that abrupt speaker changes are normal.

In [ ]:
rng = np.random.default_rng(SIM["seed"])
MIN_MIX = SIM["min_clean_sec"]

def _fit(w, n):
    if len(w) >= n:
        s = rng.integers(0, len(w) - n + 1); return w[s:s + n]
    return np.tile(w, int(np.ceil(n / max(1, len(w)))))[:n]

def build_clean(idx, max_sec=SIM["dur_sec"], tries=30):
    best = None
    for _ in range(tries):
        c = idx[rng.integers(0, len(idx))]
        w, _ = sf.read(c["path"], dtype="float32")
        if w.ndim > 1: w = w.mean(axis=1)
        if best is None or len(w) > len(best): best = w
        if len(w) >= MIN_MIX * CFG["sr"]: return w[:int(max_sec * CFG["sr"])]
    return best[:int(max_sec * CFG["sr"])] if best is not None else None

def scale_to_snr(clean_seg, noise_seg, snr_db, ref_pow=None):
    pc = float(np.mean(clean_seg ** 2))
    if ref_pow is not None: pc = max(pc, 0.1 * ref_pow)   # event may land on a pause
    pn = float(np.mean(noise_seg ** 2)) + 1e-10
    return noise_seg * math.sqrt(max(pc, 1e-10) / (pn * (10 ** (snr_db / 10.0))))

def simulate_one(cidx, nidx):
    clean = build_clean(cidx)
    if clean is None or len(clean) < int(MIN_MIX * CFG["sr"]): return None, None, []
    n = len(clean); pk = np.abs(clean).max()
    if pk > 0: clean = clean / pk * rng.uniform(0.3, 0.85)
    mix = clean.copy(); ref_pow = float(np.mean(clean ** 2))
    placed, occ = [], []
    for _ in range(int(rng.integers(SIM["events_per_mix"][0], SIM["events_per_mix"][1] + 1))):
        nb = nidx[rng.integers(0, len(nidx))]
        w, _ = sf.read(nb["path"], dtype="float32")
        if w.ndim > 1: w = w.mean(axis=1)
        L = int(min(len(w) / CFG["sr"], SIM["max_noise_sec"]) * CFG["sr"])
        if L < int(SIM["min_noise_sec"] * CFG["sr"]) or L >= n: continue
        s = None
        for _t in range(20):
            cand = int(rng.integers(0, n - L))
            if all(cand + L <= a or cand >= b_ for a, b_ in occ): s = cand; break
        if s is None: continue
        snr = float(rng.uniform(*SIM["snr_db"]))
        seg = scale_to_snr(clean[s:s + L], _fit(w, L), snr, ref_pow)
        f = min(int(0.02 * CFG["sr"]), L // 2)
        if f > 0:
            seg[:f] *= np.linspace(0, 1, f); seg[-f:] *= np.linspace(1, 0, f)
        mix[s:s + L] += seg; occ.append((s, s + L))
        placed.append({"onset": round(s / CFG["sr"], 3), "offset": round((s + L) / CFG["sr"], 3),
                       "cat": int(nb["cat"]), "snr_db": round(snr, 2)})
    m = np.abs(mix).max()
    if m > 0.99: mix, clean = mix / m * 0.99, clean / m * 0.99
    return mix.astype(np.float32), clean.astype(np.float32), sorted(placed, key=lambda e: e["onset"])

records = []
for i in tqdm(range(N_MIX)):
    cid = f"sim{i:06d}"
    mix, clean, ev = simulate_one(clean_idx, noise_idx)
    if mix is None or not ev: continue
    sf.write(MIX_DIR / f"{cid}.wav", mix, CFG["sr"])
    sf.write(REF_DIR / f"{cid}.wav", clean, CFG["sr"])
    records.append({"id": cid, "mix": str(MIX_DIR / f"{cid}.wav"),
                    "clean": str(REF_DIR / f"{cid}.wav"), "oracle_events": ev})
assert records, "no mixtures - clean bank has no spans over SIM['min_clean_sec']"
with open(T2_MANIFEST, "w") as f:
    for r in records: f.write(json.dumps(r) + "\n")
print(len(records), "mixtures |", sum(f.stat().st_size for f in MIX_DIR.glob('*.wav')) / 1e9, "GB")

---## 11. Run Track 1 over the mixturesThis is what connects the tracks. Track 2 trains on **predicted** conditioning — with its realmisses and false alarms — not oracle spans. Train on oracle and the model assumes perfecttimestamps, then collapses when it meets the detector at test time.The oracle spans stay in the manifest for the ablation in Section 14.

In [ ]:
for r in tqdm(records):
    export_track1(r["id"], sf.read(r["mix"], dtype="float32")[0])

def iou(a, b):
    inter = max(0.0, min(a[1], b[1]) - max(a[0], b[0]))
    union = max(a[1], b[1]) - min(a[0], b[0])
    return inter / union if union > 0 else 0.0

hits = tot = fp = 0
for r in records[:500]:
    pred = [tuple(e) for e in load_cond(r["id"])["events"]]; used = set()
    for e in r["oracle_events"]:
        tot += 1
        for j, p in enumerate(pred):
            if j not in used and iou(p, (e["onset"], e["offset"])) >= 0.3:
                hits += 1; used.add(j); break
    fp += len(pred) - len(used)
print(f"detector recall @IoU0.3 on sim: {hits/max(1,tot):.3f} | false alarms: {fp}")
print("If this is far below your val F1, the simulation is off-distribution - check the SNR "
      "range and the noise-bank duration profile before trusting Track 2 numbers.")

---## 12. Track 2 dataset

In [ ]:
T2_NFFT, T2_HOP = 512, 128
T2_FRAMES = N_SAMP // T2_HOP + 1
_win = torch.hann_window(T2_NFFT).to(device)

def t2_stft(x):
    return torch.stft(x, T2_NFFT, T2_HOP, window=_win, return_complex=True)

def t2_istft(X, n):
    return torch.istft(X, T2_NFFT, T2_HOP, window=_win, length=n)

class T2Dataset(Dataset):
    def __init__(self, recs, use_oracle=False):
        self.recs, self.use_oracle = recs, use_oracle
    def __len__(self): return len(self.recs)
    def _oracle(self, r, T):
        c = np.zeros((8, T), dtype=np.float32)
        for e in r["oracle_events"]:
            a, b_ = int(e["onset"] / FRAME_SEC), int(e["offset"] / FRAME_SEC)
            c[0, a:b_] = 1.0
            if e["cat"] >= 0: c[1 + e["cat"], a:b_] = 1.0
        return c
    def __getitem__(self, i):
        r = self.recs[i]
        mix, _ = sf.read(r["mix"], dtype="float32")
        clean, _ = sf.read(r["clean"], dtype="float32")
        mix, clean = mix[:N_SAMP], clean[:N_SAMP]
        if len(mix) < N_SAMP:
            mix = np.pad(mix, (0, N_SAMP - len(mix))); clean = np.pad(clean, (0, N_SAMP - len(clean)))
        cond = self._oracle(r, FRAMES_OUT) if self.use_oracle else load_cond(r["id"])["post"]
        if cond.shape[1] < FRAMES_OUT:
            cond = np.pad(cond, ((0, 0), (0, FRAMES_OUT - cond.shape[1])))
        cond = cond[:, :FRAMES_OUT]
        cond = np.concatenate([cond, (cond[0:1] >= 0.5).astype(np.float32)], axis=0)  # (9, T)
        return dict(mix=torch.from_numpy(mix.astype(np.float32)),
                    clean=torch.from_numpy(clean.astype(np.float32)),
                    cond=torch.from_numpy(cond))

split = int(0.95 * len(records))
t2_train_recs, t2_val_recs = records[:split], records[split:]
t2_train_dl = DataLoader(T2Dataset(t2_train_recs), batch_size=T2_BS, shuffle=True,
                         num_workers=2, drop_last=True)
t2_val_dl = DataLoader(T2Dataset(t2_val_recs), batch_size=T2_BS, shuffle=False, num_workers=2)
bb = next(iter(t2_train_dl))
print({k: tuple(v.shape) for k, v in bb.items()}, "| stft frames", T2_FRAMES)

---## 12b. SraVaani encoder — discovery and feature extractionWe load the exported TorchScript graph directly and skip the HF wrapper, the custom modelingcode, the processor and SentencePiece entirely. The repo contents make this straightforward:- `model-asr.fp16.ts` (908 MB) is the encoder, `forward(audio_signal, length)`- `config.json` gives `feat_in=128` and `subsampling_factor=8` — the front-end is **128 mels**,  not the NeMo default of 80- `preproc.pt` carries the exact `window`, mel filterbank `fb` and `params`, so the front-end is  reproduced rather than guessed, using the `_normalize_per_feature` routine copied verbatim  from `processing_sravaani.py`Two repo quirks worth knowing, since both cost time: `SraVaaniProcessor.from_pretrained` does`os.path.join(path, "tokenizer.model")`, so it must be handed a **local directory** — passing therepo id makes it search for a folder literally named `ARTPARK-IISc/SraVaani-1.0`. And the`AutoModel` wrapper leaves `.encoder` as `None` until `transcribe()` runs, which is why everyattempt to reach it through the wrapper found nothing.If anything still fails the run **falls back to WavLM automatically** rather than dying.Two facts to keep in view:- **80 ms frames.** FastConformer subsamples 8x. We upsample to the 20 ms grid for conditioning.  That is legitimate here (conditioning wants semantics, not boundaries) but it is why this  encoder is *not* used for Track 1, where the tolerance floor is 50 ms.- **Frozen, always.** 430M parameters, and we have 4k mixtures. Fine-tuning it would overfit long  before it helped.

In [ ]:
import os, glob

class SraVaaniFeatures(nn.Module):
    # Bypasses the HF wrapper entirely and loads the exported TorchScript graph.
    #
    # Three facts from the repo that make this work:
    #   * model-asr.fp16.ts is the encoder, forward(audio_signal, length) -> (enc, len)
    #   * config.json says feat_in=128 (NOT 80) and subsampling_factor=8
    #   * preproc.pt carries the exact `window`, mel filterbank `fb` and `params`, so the
    #     front-end can be reproduced exactly instead of guessed
    #
    # Their SraVaaniProcessor.from_pretrained does os.path.join(path, "tokenizer.model"),
    # so it must be given a LOCAL directory - passing the repo id makes it look for a
    # folder literally named "ARTPARK-IISc/SraVaani-1.0". snapshot_download gives us that path.
    def __init__(self, repo=SRAVAANI_REPO, token=None):
        super().__init__()
        from huggingface_hub import snapshot_download
        path = snapshot_download(repo, token=token)
        print(f"   snapshot: {path}")

        ts = sorted(glob.glob(os.path.join(path, "*.ts")))
        if not ts:
            raise RuntimeError(f"no TorchScript graph in {path}")
        self.enc = torch.jit.load(ts[0], map_location="cpu").eval()
        print(f"   encoder: {os.path.basename(ts[0])}")

        self.pdtype = torch.float32
        for p in self.enc.parameters():
            self.pdtype = p.dtype; break
        for p in self.enc.parameters():
            p.requires_grad_(False)

        pp = torch.load(os.path.join(path, "preproc.pt"), map_location="cpu",
                        weights_only=False)
        self.register_buffer("win", pp["window"].float(), persistent=False)
        self.register_buffer("fb", pp["fb"].float(), persistent=False)
        prm = pp.get("params", {}) or {}
        print(f"   preproc params: {dict(prm)}")
        self.n_fft = int(prm.get("n_fft", 512))
        self.hop = int(prm.get("hop_length", prm.get("hop", 160)))
        self.preemph = float(prm.get("preemph", 0.97) or 0.0)
        self.guard = float(prm.get("log_zero_guard_value", 2 ** -24))
        self.norm_c = float(prm.get("normalize_constant", 1e-5))
        self.n_mels = self.fb.shape[0] if self.fb.dim() == 2 else 128
        print(f"   front-end: exact filterbank from preproc.pt "
              f"({self.n_mels} mels, n_fft {self.n_fft}, hop {self.hop})")
        self._call = None
        self.eval()

    @staticmethod
    def _norm_per_feature(x, seq_len, constant):
        # verbatim from processing_sravaani.py
        B, _, T = x.shape
        steps = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        valid = steps < seq_len.unsqueeze(1)
        denom = valid.sum(dim=1)
        mean = torch.where(valid.unsqueeze(1), x, torch.zeros_like(x)).sum(dim=2) \
               / denom.unsqueeze(1)
        var = torch.sum(torch.where(valid.unsqueeze(1), x - mean.unsqueeze(2),
                                    torch.zeros_like(x)) ** 2, dim=2) \
              / (denom.unsqueeze(1) - 1.0)
        std = torch.sqrt(var)
        std = std.masked_fill(std.isnan(), 0.0) + constant
        return (x - mean.unsqueeze(2)) / std.unsqueeze(2)

    def _features(self, wav):
        if self.preemph:
            wav = torch.cat([wav[:, :1], wav[:, 1:] - self.preemph * wav[:, :-1]], dim=1)
        X = torch.stft(wav, self.n_fft, self.hop, win_length=self.win.numel(),
                       window=self.win.to(wav.device), center=True, return_complex=True)
        mel = torch.matmul(self.fb.to(wav.device), X.abs() ** 2)      # (B, n_mels, T)
        mel = torch.log(mel + self.guard)
        lens = torch.full((mel.shape[0],), mel.shape[-1], device=mel.device,
                          dtype=torch.float32)
        return self._norm_per_feature(mel, lens, self.norm_c).to(self.pdtype)

    def _run(self, feats, lens):
        trials = [("audio_signal=,length=", lambda: self.enc(audio_signal=feats, length=lens)),
                  ("positional(f,len)",     lambda: self.enc(feats, lens))]
        if self._call is not None:
            trials = [t for t in trials if t[0] == self._call]
        errs = []
        for name, fn in trials:
            try:
                out = fn()
            except Exception as e:
                errs.append(f"{name}: {type(e).__name__}: {str(e)[:120]}"); continue
            self._call = name
            return out[0] if isinstance(out, (tuple, list)) else out
        raise RuntimeError("encoder call failed:\n  " + "\n  ".join(errs))

    @torch.no_grad()
    def forward(self, wav):
        feats = self._features(wav)
        lens = torch.full((feats.shape[0],), feats.shape[-1], dtype=torch.long,
                          device=wav.device)
        return self._run(feats, lens).transpose(1, 2).float()   # (B, D, T) -> (B, T, D)

def build_t2_encoder():
    # returns (module, out_dim). module(wav) -> (B, T, D) at whatever the encoder's frame rate is
    if T2_ENCODER == "sravaani":
        tok = HF_TOKEN if "HF_TOKEN" in globals() else None
        try:
            enc = SraVaaniFeatures(token=tok).to(device)
            with torch.no_grad():
                probe = enc(torch.zeros(1, CFG["sr"], device=device))
            d = probe.shape[-1]
            print(f"SraVaani OK via {enc._call}: 1.00 s -> {probe.shape[1]} frames "
                  f"({1000/probe.shape[1]:.0f} ms/frame), dim {d}")
            return enc, d
        except Exception as e:
            print(f"\nSraVaani unavailable ({type(e).__name__}: {e})")
            print("Falling back to the Track 1 WavLM encoder so the run can continue.\n")
    enc = AutoModel.from_pretrained(ENCODER_NAME)
    if DROP_LAST_LAYERS > 0:
        enc.encoder.layers = enc.encoder.layers[:-DROP_LAST_LAYERS]
    if T2_SSL_FROM_T1:
        sd = torch.load(CKPT, map_location="cpu", weights_only=False)["model"]
        w = {k[len("enc."):]: v for k, v in sd.items() if k.startswith("enc.")}
        miss, unexp = enc.load_state_dict(w, strict=False)
        print(f"WavLM from Track 1: {len(w)} tensors, {len(miss)} missing, {len(unexp)} unexpected")
    else:
        print("stock WavLM")

    class _W(nn.Module):
        def __init__(self, m):
            super().__init__(); self.m = m
        @torch.no_grad()
        def forward(self, wav):
            x = (wav - wav.mean(-1, keepdim=True)) / (wav.std(-1, keepdim=True) + 1e-5)
            return self.m(x).last_hidden_state.float()
    w_ = _W(enc).to(device)
    for p in w_.parameters(): p.requires_grad_(False)
    return w_, enc.config.hidden_size

---## 13. Track 2 model — pretrained encoder + FiLM mask headThree signals reach the mask head:1. **The mixture spectrogram** (257 bins, 8 ms hop) — what to modify.2. **Encoder frames** — SraVaani (430M FastConformer ASR, 31k h of Vaani, 80 ms frames) by   default, or the fine-tuned Track 1 WavLM (20 ms frames) via `T2_ENCODER = "wavlm"`.3. **Track 1 posteriors** (9 channels, 20 ms) — where the detector believes the events are.Signals 2 and 3 are concatenated (they share the 20 ms grid), interpolated to the STFT rate, andinjected by **FiLM at two depths**. Not concatenated at the input: input-only conditioning hasbeen measured to perform *worse* than no conditioning at all (SLICE, arXiv 2603.05302), becausethe signal is diluted before reaching the layers that use it.The encoder is frozen either way. At 430M parameters against 4k mixtures, fine-tuning it wouldoverfit long before it helped, and freezing keeps the trainable head at ~2.7M so it converges in25 epochs.**Why an ASR encoder for Track 2 specifically.** The Track 2 score is `SI-SDR + 100 x dWER_frac`,so one WER point weighs like one dB. The dominant failure mode is over-suppression that deletesphonemes. SraVaani's features encode what is phonetically load-bearing, which is exactly thesignal that should restrain the mask. The same bias makes it a poor Track 1 encoder — an ASRmodel is trained to be *invariant* to background events, which is the information Track 1 needs.

In [ ]:
class CondMaskNet(nn.Module):
    def __init__(self, encoder, enc_dim, n_freq=T2_NFFT // 2 + 1, cond_dim=9, hid=256,
                 proj=T2_SSL_PROJ):
        super().__init__()
        self.enc = encoder                      # frozen, any frame rate
        self.enc_proj = nn.Linear(enc_dim, proj)
        cdim = proj + cond_dim
        self.inp = nn.Linear(n_freq, hid)
        self.film1 = nn.Linear(cdim, 2 * hid)
        self.rnn = nn.GRU(hid, hid, 2, batch_first=True, bidirectional=True, dropout=0.1)
        self.film2 = nn.Linear(cdim, 2 * 2 * hid)
        self.mid = nn.Linear(2 * hid, 2 * hid)
        self.out = nn.Linear(2 * hid, n_freq)

    def forward(self, wav, mag_log, cond, T_stft):
        e = self.enc_proj(self.enc(wav))                     # (B, Te, proj)
        c = cond                                             # (B, 9, Tc) @ 20 ms
        # Resample BOTH to the STFT frame rate. SraVaani is 80 ms and WavLM is 20 ms, so the
        # encoder stream is upsampled by 10x or 2.5x respectively; the posteriors are always
        # 20 ms. Doing it independently avoids forcing one onto the other's grid first.
        e = F.interpolate(e.transpose(1, 2), size=T_stft, mode="linear",
                          align_corners=False).transpose(1, 2)
        c = F.interpolate(c, size=T_stft, mode="linear", align_corners=False).transpose(1, 2)
        ctx = torch.cat([e, c], dim=-1)
        h = self.inp(mag_log)
        g, b_ = self.film1(ctx).chunk(2, -1); h = h * (1 + g) + b_
        h, _ = self.rnn(h)
        g, b_ = self.film2(ctx).chunk(2, -1); h = h * (1 + g) + b_
        h = F.relu(self.mid(h))
        return torch.sigmoid(self.out(h))

def enhance_batch(model, mix, cond):
    X = t2_stft(mix)                                          # (B, F, T) complex
    mag_log = torch.log1p(X.abs()).transpose(1, 2)            # (B, T, F)
    m = model(mix, mag_log, cond, X.shape[-1]).transpose(1, 2)
    return t2_istft(X * m, mix.shape[-1]), m

_enc, _dim = build_t2_encoder()
t2 = CondMaskNet(_enc, _dim).to(device)
trainable = sum(p.numel() for p in t2.parameters() if p.requires_grad)
print(f"{sum(p.numel() for p in t2.parameters())/1e6:.1f}M total, {trainable/1e6:.2f}M trainable "
      f"(encoder frozen)")
with torch.no_grad():
    _e, _m = enhance_batch(t2, bb["mix"].to(device), bb["cond"].to(device))
print("est", tuple(_e.shape), "mask", tuple(_m.shape))

---## 14. Train Track 2Loss is **SI-SDR in the time domain** — the leaderboard metric itself, not a proxy. The mask isapplied to the complex STFT and reconstructed with the mixture's phase, so gradients flow fromthe waveform back through the mask into the FiLM layers.

In [ ]:
def si_sdr_loss(est, ref, eps=1e-8):
    est = est - est.mean(-1, keepdim=True); ref = ref - ref.mean(-1, keepdim=True)
    a = (est * ref).sum(-1, keepdim=True) / ((ref * ref).sum(-1, keepdim=True) + eps)
    t = a * ref; n = est - t
    return -(10 * torch.log10(((t ** 2).sum(-1) + eps) / ((n ** 2).sum(-1) + eps))).mean()

def si_sdr_np(est, ref, eps=1e-9):
    est = est - est.mean(); ref = ref - ref.mean()
    a = np.dot(est, ref) / (np.dot(ref, ref) + eps)
    t = a * ref; n = est - t
    return 10 * np.log10((t ** 2).sum() + eps) - 10 * np.log10((n ** 2).sum() + eps)

@torch.no_grad()
def t2_eval(model, dl):
    model.eval(); ins, outs = [], []
    for batch in dl:
        mix, clean = batch["mix"].to(device), batch["clean"].to(device)
        est, _ = enhance_batch(model, mix, batch["cond"].to(device))
        for e, c, m in zip(est.float().cpu().numpy(), clean.cpu().numpy(), mix.cpu().numpy()):
            ins.append(si_sdr_np(m, c)); outs.append(si_sdr_np(e, c))
    return dict(si_sdr_in=float(np.mean(ins)), si_sdr_out=float(np.mean(outs)),
                si_sdri=float(np.mean(np.array(outs) - np.array(ins))))

params = [p for p in t2.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=T2_EPOCHS * max(1, len(t2_train_dl)))
best_t2 = -1e9
for ep in range(T2_EPOCHS):
    t2.train()
    t2.enc.eval()      # encoder stays in inference mode: frozen norm/dropout
    tot = 0.0; n = 0; t0 = time.time()
    for batch in t2_train_dl:
        mix, clean = batch["mix"].to(device), batch["clean"].to(device)
        est, _ = enhance_batch(t2, mix, batch["cond"].to(device))
        loss = si_sdr_loss(est, clean)
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(params, 5.0)
        opt.step(); sched.step()
        tot += float(loss.detach()); n += 1
    m = t2_eval(t2, t2_val_dl)
    print(f"ep{ep:02d} [{(time.time()-t0)/60:.1f}m] loss {tot/max(1,n):.3f} | "
          f"SI-SDR {m['si_sdr_in']:.2f} -> {m['si_sdr_out']:.2f} ({m['si_sdri']:+.2f} dB)")
    if m["si_sdri"] > best_t2:
        best_t2 = m["si_sdri"]
        torch.save({"model": t2.state_dict(), "si_sdri": best_t2,
                    "ssl_from_t1": T2_SSL_FROM_T1}, CKPT_T2)
print(f"\nbest SI-SDRi {best_t2:+.2f} dB")

---## 15. Ablations — is any of this earning its place?Four readings, all on the same trained model:- **predicted** — what you will submit- **oracle** — the ceiling if Track 1 were perfect; the gap says which track deserves more time- **zero** — Track 1 signal removed. If this matches predicted, the conditioning does nothing and  the FiLM wiring is what to fix.- **gated Wiener** — a zero-training DSP reference. If the trained model cannot beat it, the  model is not earning its runtime.

In [ ]:
import scipy.signal as ss

t2.load_state_dict(torch.load(CKPT_T2, map_location=device, weights_only=False)["model"])
t2.eval()

@torch.no_grad()
def ablate(recs, mode):
    dl = DataLoader(T2Dataset(recs, use_oracle=(mode == "oracle")), batch_size=T2_BS,
                    shuffle=False, num_workers=2)
    ins, outs = [], []
    for batch in dl:
        mix, clean = batch["mix"].to(device), batch["clean"].to(device)
        cond = batch["cond"].to(device)
        if mode == "zero": cond = torch.zeros_like(cond)
        est, _ = enhance_batch(t2, mix, cond)
        for e, c, m in zip(est.float().cpu().numpy(), clean.cpu().numpy(), mix.cpu().numpy()):
            ins.append(si_sdr_np(m, c)); outs.append(si_sdr_np(e, c))
    return float(np.mean(np.array(outs) - np.array(ins)))

N_FFT_W, HOP_W = 512, 128
def gated_wiener(wav, events, sr=CFG["sr"], alpha=1.6, floor=0.08, ctx=0.5, fade=0.03):
    if len(events) == 0: return wav.astype(np.float32)
    X = ss.stft(wav, nperseg=N_FFT_W, noverlap=N_FFT_W - HOP_W)[2]
    P = np.abs(X) ** 2; T = X.shape[1]; tp = HOP_W / sr; G = np.ones_like(P)
    for on, off in events:
        a, b_ = int(on / tp), min(T, int(off / tp))
        if b_ <= a: continue
        c = np.concatenate([P[:, max(0, a - int(ctx / tp)):a],
                            P[:, b_:min(T, b_ + int(ctx / tp))]], axis=1)
        if c.shape[1] < 3: c = P[:, a:b_]
        npsd = np.maximum(P[:, a:b_].mean(1, keepdims=True) - c.mean(1, keepdims=True), 0.0)
        G[:, a:b_] = np.maximum(1.0 - alpha * npsd / (P[:, a:b_] + 1e-10), floor)
    k = max(1, int(fade / tp))
    G = ss.convolve2d(G, np.ones((1, k)) / k, mode="same", boundary="symm")
    y = ss.istft(X * G, nperseg=N_FFT_W, noverlap=N_FFT_W - HOP_W)[1]
    return np.pad(y, (0, max(0, len(wav) - len(y))))[:len(wav)].astype(np.float32)

def wiener_ref(recs):
    d = []
    for r in recs:
        mix, _ = sf.read(r["mix"], dtype="float32"); clean, _ = sf.read(r["clean"], dtype="float32")
        est = gated_wiener(mix, load_cond(r["id"])["events"])
        L = min(len(est), len(clean), len(mix))
        d.append(si_sdr_np(est[:L], clean[:L]) - si_sdr_np(mix[:L], clean[:L]))
    return float(np.mean(d))

print(f"{'predicted cond':>18}: {ablate(t2_val_recs, 'predicted'):+.2f} dB")
print(f"{'oracle cond':>18}: {ablate(t2_val_recs, 'oracle'):+.2f} dB")
print(f"{'zeroed cond':>18}: {ablate(t2_val_recs, 'zero'):+.2f} dB")
print(f"{'gated Wiener (DSP)':>18}: {wiener_ref(t2_val_recs):+.2f} dB")

---## 16. Track 2 submissionEnhanced 16 kHz mono WAVs, one per test clip, filenames preserved. Track 1 runs first on eachclip to produce the conditioning, exactly as in training.

In [ ]:
T2_TEST_DIR = Path("/kaggle/input/indoml-track2-test")
SUB_DIR = WORK / "t2_submission"; SUB_DIR.mkdir(exist_ok=True)
TEST_COND = WORK / "t1_cond_test"; TEST_COND.mkdir(exist_ok=True)

@torch.no_grad()
def enhance_file(wav, clip_id):
    post = load_cond(clip_id, TEST_COND)["post"]
    cond = np.concatenate([post, (post[0:1] >= 0.5).astype(np.float32)], axis=0)
    x = torch.from_numpy(wav).unsqueeze(0).to(device)
    c = torch.from_numpy(cond).unsqueeze(0).to(device)
    est, _ = enhance_batch(t2, x, c)
    y = est[0].float().cpu().numpy()
    peak = np.abs(y).max()
    if peak > 0.99: y = y / peak * 0.99            # avoid clipping on write
    return y.astype(np.float32)

if T2_TEST_DIR.exists():
    files = sorted([p for p in T2_TEST_DIR.rglob("*") if p.suffix.lower() in AUD])
    print(len(files), "test clips")
    for p in tqdm(files):
        w = read_audio(p)
        export_track1(p.stem, w, cond_dir=TEST_COND)                 # 1) detect
        sf.write(SUB_DIR / f"{p.stem}.wav", enhance_file(w, p.stem), CFG["sr"])  # 2) suppress
    import shutil
    shutil.make_archive(str(WORK / "submission_track2"), "zip", SUB_DIR)
    n = len(list(SUB_DIR.glob("*.wav")))
    print(f"{n} wavs -> {WORK/'submission_track2.zip'}")
    chk, _ = sf.read(str(next(SUB_DIR.glob('*.wav'))), dtype="float32")
    print(f"sanity: first output {chk.shape}, 16 kHz mono, peak {np.abs(chk).max():.3f}")
else:
    print(f"{T2_TEST_DIR} not found - accept the Track 2 terms on Codabench, download "
          f"input_data from the Files tab, attach as a Kaggle Dataset, re-run.")

---## 17. Reading the results, and what to do next**Read Section 15 before anything else.**- `zeroed ~= predicted` -> conditioning is doing nothing. Check that Section 11's detector recall  is not near zero, then that `cond` is not all zeros in the batch.- `predicted << oracle` -> Track 1 is the bottleneck. More gold, more epochs, better encoder.- `predicted <= gated Wiener` -> the trained model is not earning its runtime. Usually means too  few mixtures; raise `N_MIX`.**Then, in order of payoff:**1. **`BUDGET = "full"`** — more data for both stages.2. **Unfreeze the SSL encoder** for the last few epochs (`T2_SSL_FROZEN = False`, LR ~1e-5).   Costs roughly 2x per step; try it only once the frozen version works.3. **Check dWER before trusting an SI-SDR gain.** Track 2 scores `SI-SDR + 100 x dWER_fraction`,   so 1 WER point weighs like 1 dB. Over-suppression deletes phonemes. Run IndicWhisper or   AI4Bharat IndicConformer over the enhanced val audio and compare against the clean reference.4. **Complex mask instead of magnitude.** Reconstructing with the mixture phase caps SI-SDR;   predicting a complex ratio mask lifts that ceiling.5. **Ablate `T2_ENCODER`.** SraVaani vs the Track 1 WavLM is a clean one-flag experiment, and   the comparison is a genuine result: an in-domain ASR encoder against a noise-aware SSL one,   on a metric that weights word errors as heavily as signal quality.**State plainly in your writeup:** the clean reference is Vaani speech with no *annotated* noiseevent, not truly clean audio, so SI-SDR is measured against a pseudo-clean target. Switch to the4 h validation set the moment it is released.